In [1]:
import subprocess
import os
import time

In [3]:
# Parameters for Laptop Processing
SNAP_Engine_Path   = 'C:/Users/jepir/Desktop/snap-engine-master/'
Unwrap_Directory   = 'C:/SNAPHU/Prod/'
Sub_Swaths         = ['IW1', 'IW2', 'IW3']
Area_Code          = 'PortugalNorth_A_'
#Sub_Swaths         = ['IW3']
Products_Directory = 'C:/Users/jepir/Desktop/S1-SLC/Raw_Data/'
Output_Directory_1 = 'C:/Users/jepir/Desktop/S1-SLC/Process_' + Area_Code[:-1] + '/STAGE_1/'
Output_Directory_2 = 'C:/Users/jepir/Desktop/S1-SLC/Process_' + Area_Code[:-1] + '/STAGE_2/'
Output_Directory_3 = 'C:/Users/jepir/Desktop/S1-SLC/Process_' + Area_Code[:-1] + '/STAGE_3/'
Output_Directory_4 = 'C:/Users/jepir/Desktop/S1-SLC/Process_' + Area_Code[:-1] + '/STAGE_4/'
Output_Directory_5 = 'C:/Users/jepir/Desktop/S1-SLC/Process_' + Area_Code[:-1] + '/STAGE_5/'
Output_Directory_6 = 'C:/Users/jepir/Desktop/S1-SLC/Process_' + Area_Code[:-1] + '/STAGE_6/'
Output_Directory_7 = 'C:/Users/jepir/Desktop/S1-SLC/Process_' + Area_Code[:-1] + '/STAGE_7/'
Output_Directory_8 = 'C:/Users/jepir/Desktop/S1-SLC/Process_' + Area_Code[:-1] + '/STAGE_8/'
Output_Directory_9 = 'C:/Users/jepir/Desktop/S1-SLC/Process_' + Area_Code[:-1] + '/STAGE_9/'

# Parameters for External Disk Processing
#SNAP_Engine_Path   = 'C:/Users/jepir/Desktop/snap-engine-master/'
#Sub_Swaths         = ['IW1', 'IW2', 'IW3']
#Products_Directory = 'E:/Test/Raw_Data/'
#Output_Directory_1 = 'E:/Test/Process/STAGE_1/'
#Output_Directory_2 = 'E:/Test/Process/STAGE_2/'
#Output_Directory_3 = 'E:/Test/Process/STAGE_3/'
#Output_Directory_4 = 'E:/Test/Process/STAGE_4/'

# Months Dictionary
Months_Dict = {
    '05':'May',
    '06':'Jun',
    '07':'Jul',
    '08':'Aug',
    '09':'Sep',
    '10':'Oct'
}

In [4]:
# Get Products to Preprocess
Products      = os.listdir(Products_Directory)
Products_Dict = dict()
# Organise products for process
for i in range(0,len(Products),2):
    Product_1_Date           = Products[i].split('_')[5].split('T')[0]
    Product_2_Date           = Products[i+1].split('_')[5].split('T')[0]
    Products_ID              = Products[i].split('_')[-1].split('.')[0]
    New_Product_Name_Generic = 'S1A_' + Product_1_Date + '_' + Product_2_Date + '_' + Products_ID + '_Stack_'
    # Save on the dictionary
    Products_Dict[New_Product_Name_Generic] = [Products[i], Products[i+1]]    

In [ ]:
# Count time
Time_Process_Start = time.perf_counter()
# Execute Stage 1
Stage1_OutputPaths = []
for key in Products_Dict.keys():
    Time_Process_Start_Command = time.perf_counter()
    # Execute the command for all subswats
    for Sub_Swath in Sub_Swaths:
        Input_Path_1 = Products_Directory + Products_Dict[key][0]
        Input_Path_2 = Products_Directory + Products_Dict[key][1]
        Output_Path   = Output_Directory_1 + Sub_Swath + '/' + key + Sub_Swath + '.dim'
        Stage1_OutputPaths.append(Output_Path)
        print(Output_Path)
        print('------------------')
        #Execute Command for Stage 1
        command = 'gpt 1_S1-SLC_Backgeocoding_GPT.xml -PInput_1=\"' + Input_Path_1 + '\" -PInput_2=\"' + Input_Path_2 + '\" -POutput=\"' + Output_Path + '\" -PSub_Swath=\"' + Sub_Swath + '\"'
        # Start process
        process = subprocess.Popen(
            command,
            shell=True,
            cwd=SNAP_Engine_Path,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,  # merge stdout and stderr
            text=True
        )
        # Stream output line by line
        for line in process.stdout:
            print(line, end="")   # already has \n, so no extra newline
        print('------------------')
    # Compute processing time
    Time_Process_Ends = time.perf_counter()
    Processing_Time   = Time_Process_Ends - Time_Process_Start_Command
    # Convert to h:m:s
    hours, rem       = divmod(Processing_Time, 3600)
    minutes, seconds = divmod(rem, 60)
    print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")
# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

In [ ]:
# Count time
Time_Process_Start = time.perf_counter()
# Execute Stage 2
Stage2_OutputPaths = []
for Path in Stage1_OutputPaths:
    Time_Process_Start_Command = time.perf_counter()
    Sub_Swath    = Path.split('/')[-1].split('_')[-1].split('.')[0]
    #if Sub_Swath != 'IW3':
    #    continue
    Input_Path_1 = Path
    Output_Path  = Output_Directory_2 + Sub_Swath + '/' + Path.split('/')[-1].rsplit('_', 1)[0] + '_esd_' + Sub_Swath + '.dim'
    Stage2_OutputPaths.append(Output_Path)
    print(Output_Path)
    print('------------------')
    #Execute Command for Stage 2
    command = 'gpt 2_S1-SLC_ESD.xml -PInput_1=\"' + Input_Path_1 + '\" -POutput=\"' + Output_Path + '\"'
    # Start process
    process = subprocess.Popen(
        command,
        shell=True,
        cwd=SNAP_Engine_Path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge stdout and stderr
        text=True
    )
    # Stream output line by line
    for line in process.stdout:
        print(line, end="")   # already has \n, so no extra newline
    print('------------------')
    # Compute processing time
    Time_Process_Ends = time.perf_counter()
    Processing_Time   = Time_Process_Ends - Time_Process_Start_Command
    # Convert to h:m:s
    hours, rem       = divmod(Processing_Time, 3600)
    minutes, seconds = divmod(rem, 60)
    print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")
# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

In [11]:
# Count time
Time_Process_Start = time.perf_counter()
# Execute Stage 3
Stage3_OutputPaths = []
for Path in Stage2_OutputPaths:
    Time_Process_Start_Command = time.perf_counter()
    Sub_Swath    = Path.split('/')[-1].split('_')[-1].split('.')[0]
    #if Sub_Swath != 'IW3':
    #    continue
    Input_Path_1 = Path
    Output_Path  = Output_Directory_3 + Sub_Swath + '/' + Path.split('/')[-1].rsplit('_', 1)[0] + '_ifg_deb_' + Sub_Swath + '.dim'
    Stage3_OutputPaths.append(Output_Path)
    print(Output_Path)
    print('------------------')
    #Execute Command for Stage 3
    command = 'gpt 3_S1-SLC_Deburst.xml -PInput_1=\"' + Input_Path_1 + '\" -POutput=\"' + Output_Path + '\"'
    # Start process
    process = subprocess.Popen(
        command,
        shell=True,
        cwd=SNAP_Engine_Path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge stdout and stderr
        text=True
    )
    # Stream output line by line
    for line in process.stdout:
        print(line, end="")   # already has \n, so no extra newline
    print('------------------')
    # Compute processing time
    Time_Process_Ends = time.perf_counter()
    Processing_Time   = Time_Process_Ends - Time_Process_Start_Command
    # Convert to h:m:s
    hours, rem       = divmod(Processing_Time, 3600)
    minutes, seconds = divmod(rem, 60)
    print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")
# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

C:/Users/jepir/Desktop/S1-SLC/Process_PortugalNorth_A/STAGE_3/IW2/S1A_20200808_20200820_821D_Stack_esd_ifg_deb_IW2.dim
------------------
OpenJDK 64-Bit Server VM warning: Options -Xverify:none and -noverify were deprecated in JDK 13 and will likely be removed in a future release.
INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
Executing processing graph
1220 [main] INFO hdf.hdflib.HDFLibrary - HDF4 library: 
1220 [main] INFO hdf.hdflib.HDFLibrary -  successfully loaded.
1224 [main] INFO hdf.hdf5lib.H5 - HDF5 library: 
1224 [main] INFO hdf.hdf5lib.H5 -  successfully loaded.
-- org.jblas INFO Starting temp DLL cleanup task.
-- org.jblas INFO Deleted 28 unused temp DLL libraries from C:\Users\jepir\AppData\Local\Temp
....10%....20%....30%....40%....50%....60%....70%....80%....90% done.
------------------
Elapse

In [12]:
Stage3_OutputPaths = []
for Path in os.listdir(Output_Directory_3 + 'IW1'):
    if Path.endswith('.dim'):
        Stage3_OutputPaths.append(Output_Directory_3 + 'IW1/' + Path)
for Path in os.listdir(Output_Directory_3 + 'IW2'):
    if Path.endswith('.dim'):
        Stage3_OutputPaths.append(Output_Directory_3 + 'IW2/' + Path)
for Path in os.listdir(Output_Directory_3 + 'IW3'):
    if Path.endswith('.dim'):
        Stage3_OutputPaths.append(Output_Directory_3 + 'IW3/' + Path)

In [13]:
# Count time
Time_Process_Start = time.perf_counter()
# Execute Stage 4
Stage4_OutputPaths = []
for Path in Stage3_OutputPaths:
    Time_Process_Start_Command = time.perf_counter()
    Sub_Swath    = Path.split('/')[-1].split('_')[-1].split('.')[0]
    # Define the number of Looks depending on the Sub Swath
    if Sub_Swath == 'IW1':
        Range_Looks = str(6)
    elif Sub_Swath == 'IW2':
        Range_Looks = str(6)
    elif Sub_Swath == 'IW3':
        Range_Looks = str(7)
    Input_Path_1 = Path
    Output_Path  = Output_Directory_4 + Sub_Swath + '/' + Path.split('/')[-1].rsplit('_', 1)[0] + '_ml_flt_' + Sub_Swath + '.dim'
    Stage4_OutputPaths.append(Output_Path)
    print(Output_Path)
    print('------------------')
    #Execute Command for Stage 4
    command = 'gpt 4_S1-SLC_Goldstein.xml -PInput_1=\"' + Input_Path_1 + '\" -PRange_Looks=\"' + Range_Looks + '\" -POutput=\"' + Output_Path + '\"'
    # Start process
    process = subprocess.Popen(
        command,
        shell=True,
        cwd=SNAP_Engine_Path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge stdout and stderr
        text=True
    )
    # Stream output line by line
    for line in process.stdout:
        print(line, end="")   # already has \n, so no extra newline
    print('------------------')
    # Compute processing time
    Time_Process_Ends = time.perf_counter()
    Processing_Time   = Time_Process_Ends - Time_Process_Start_Command
    # Convert to h:m:s
    hours, rem       = divmod(Processing_Time, 3600)
    minutes, seconds = divmod(rem, 60)
    print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")
# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

C:/Users/jepir/Desktop/S1-SLC/Process_PortugalNorth_A/STAGE_4/IW1/S1A_20200609_20200621_CD18_Stack_esd_ifg_deb_ml_flt_IW1.dim
------------------
OpenJDK 64-Bit Server VM warning: Options -Xverify:none and -noverify were deprecated in JDK 13 and will likely be removed in a future release.
INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
Executing processing graph
....10%....20%....30%....40%....50%....60%....70%....80%....90% done.
------------------
Elapsed time: 00:01:39.46
C:/Users/jepir/Desktop/S1-SLC/Process_PortugalNorth_A/STAGE_4/IW1/S1A_20200703_20200715_D467_Stack_esd_ifg_deb_ml_flt_IW1.dim
------------------
OpenJDK 64-Bit Server VM warning: Options -Xverify:none and -noverify were deprecated in JDK 13 and will likely be removed in a future release.
INFO: org.esa.snap.core.gpf.operators.tooladapter.To

In [14]:
# Count time
Time_Process_Start = time.perf_counter()
# Execute Stage 5
Stage5_OutputPaths_VV = []
Stage5_OutputPaths_VH = []
for Path in Stage4_OutputPaths:
    Time_Process_Start_Command = time.perf_counter()
    Sub_Swath    = Path.split('/')[-1].split('_')[-1].split('.')[0]
    #if Sub_Swath != 'IW3':
    #    continue
    Input_Path_1    = Path
    Output_Path_VV  = Output_Directory_5 + Sub_Swath + '/' + Path.split('/')[-1].rsplit('_', 1)[0] + '_VV_' + Sub_Swath + '.dim'
    Output_Path_VH  = Output_Directory_5 + Sub_Swath + '/' + Path.split('/')[-1].rsplit('_', 1)[0] + '_VH_' + Sub_Swath + '.dim'
    Stage5_OutputPaths_VV.append(Output_Path_VV)
    Stage5_OutputPaths_VH.append(Output_Path_VH)
    print(Output_Path_VV)
    print('------------------')
    # Define Variables to Subset
    Dates_Code = Path.split('/')[-1].split('_')[1][6:8] + Months_Dict[Path.split('/')[-1].split('_')[1][4:6]] + Path.split('/')[-1].split('_')[1][0:4] + '_' \
               + Path.split('/')[-1].split('_')[2][6:8] + Months_Dict[Path.split('/')[-1].split('_')[2][4:6]] + Path.split('/')[-1].split('_')[2][0:4]
    Subset_VV = 'Phase_ifg_' + Sub_Swath + '_VV_' + Dates_Code + ',coh_' + Sub_Swath + '_VV_' + Dates_Code
    Subset_VH = 'Phase_ifg_' + Sub_Swath + '_VH_' + Dates_Code + ',coh_' + Sub_Swath + '_VH_' + Dates_Code
    #Execute Command for Stage 5
    command = 'gpt 5_S1-SLC_Subset.xml -PInput_1=\"' + Input_Path_1 + '\" -POutput_Path_VV=\"' + Output_Path_VV + '\" -POutput_Path_VH=\"' + Output_Path_VH + '\" -PSubset_VV=\"' + Subset_VV + '\" -PSubset_VH=\"' + Subset_VH + '\"'
    # Start process
    process = subprocess.Popen(
        command,
        shell=True,
        cwd=SNAP_Engine_Path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge stdout and stderr
        text=True
    )
    # Stream output line by line
    for line in process.stdout:
        print(line, end="")   # already has \n, so no extra newline
    print('------------------')
    # Compute processing time
    Time_Process_Ends = time.perf_counter()
    Processing_Time   = Time_Process_Ends - Time_Process_Start_Command
    # Convert to h:m:s
    hours, rem       = divmod(Processing_Time, 3600)
    minutes, seconds = divmod(rem, 60)
    print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")
# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

C:/Users/jepir/Desktop/S1-SLC/Process_PortugalNorth_A/STAGE_5/IW1/S1A_20200609_20200621_CD18_Stack_esd_ifg_deb_ml_flt_VV_IW1.dim
------------------
OpenJDK 64-Bit Server VM warning: Options -Xverify:none and -noverify were deprecated in JDK 13 and will likely be removed in a future release.
INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
Executing processing graph
....10%....20%....30%....40%....50%....60%....70%....80%....90% done.
------------------
Elapsed time: 00:00:14.07
C:/Users/jepir/Desktop/S1-SLC/Process_PortugalNorth_A/STAGE_5/IW1/S1A_20200703_20200715_D467_Stack_esd_ifg_deb_ml_flt_VV_IW1.dim
------------------
OpenJDK 64-Bit Server VM warning: Options -Xverify:none and -noverify were deprecated in JDK 13 and will likely be removed in a future release.
INFO: org.esa.snap.core.gpf.operators.tooladap

In [15]:
Stage5_OutputPaths = []
for Path in os.listdir(Output_Directory_5 + 'IW1'):
    if Path.endswith('.dim'):
        Stage5_OutputPaths.append(Output_Directory_5 + 'IW1/' + Path)
for Path in os.listdir(Output_Directory_5 + 'IW2'):
    if Path.endswith('.dim'):
        Stage5_OutputPaths.append(Output_Directory_5 + 'IW2/' + Path)
for Path in os.listdir(Output_Directory_5 + 'IW3'):
    if Path.endswith('.dim'):
        Stage5_OutputPaths.append(Output_Directory_5 + 'IW3/' + Path)

In [16]:
# Count time
Time_Process_Start = time.perf_counter()
# Execute Stage 6
for Input_Path_1 in Stage5_OutputPaths:
    Time_Process_Start_Command = time.perf_counter()
    # ----Export----
    print('------------------')
    print(Input_Path_1)
    #Execute Command for Stage 6 - Export
    command = 'gpt 6_S1-SLC_SnaphuExport.xml -PInput_1=\"' + Input_Path_1 + '\" -POutput=\"' + Unwrap_Directory + '\"'
    # Start process
    process = subprocess.Popen(
        command,
        shell=True,
        cwd=SNAP_Engine_Path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge stdout and stderr
        text=True
    )
    for line in process.stdout:
        print(line, end="")   # already has \n, so no extra newline
    # ----Unwrap----
    # Get Unwrapping Configuration files
    Unw_Folder = Unwrap_Directory + Input_Path_1.split('/')[-1].split('.')[0]
    # Get Command VV
    with open(Unw_Folder + '/snaphu.conf', 'r') as f:
        command = [line.strip() for line in f if 'snaphu -f'.lower() in line.lower()][0][1:].lstrip()
    #Execute Unwrap command
    process = subprocess.Popen(
        command,
        shell=True,
        cwd=Unw_Folder,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge stdout and stderr
        text=True
    )
    for line in process.stdout:
        print(line, end="")   # already has \n, so no extra newline
    # Get Unwrapping files
    Unw_Path_1 = Unw_Folder + '/' + [File for File in os.listdir(Unw_Folder) if File.startswith('UnwPhase_') and File.endswith('.hdr')][0]
    # Define Output Paths
    Output_Path_1 = Output_Directory_6 + Input_Path_1.split('/')[-1].rsplit('_', 2)[0] + '_unw_' + Input_Path_1.split('/')[-1].split('_', 10)[-1]
    # Execute command
    command = 'gpt 6_S1-SLC_SnaphuImport.xml -PInput_1=\"' + Input_Path_1 + '\" -PInput_2=\"' + Unw_Path_1 + '\" -POutput=\"' + Output_Path_1 + '\"' 
    # Start process
    process = subprocess.Popen(
        command,
        shell=True,
        cwd=SNAP_Engine_Path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge stdout and stderr
        text=True
    )
    for line in process.stdout:
        print(line, end="")   # already has \n, so no extra newline
    Time_Process_Ends = time.perf_counter()
    Processing_Time   = Time_Process_Ends - Time_Process_Start_Command
    # Convert to h:m:s
    hours, rem       = divmod(Processing_Time, 3600)
    minutes, seconds = divmod(rem, 60)
    print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")
# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

------------------
C:/Users/jepir/Desktop/S1-SLC/Process_PortugalNorth_A/STAGE_5/IW1/S1A_20200609_20200621_CD18_Stack_esd_ifg_deb_ml_flt_VH_IW1.dim
OpenJDK 64-Bit Server VM warning: Options -Xverify:none and -noverify were deprecated in JDK 13 and will likely be removed in a future release.
INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
Executing processing graph
2802 [main] INFO hdf.hdflib.HDFLibrary - HDF4 library: 
2802 [main] INFO hdf.hdflib.HDFLibrary -  successfully loaded.
2852 [main] INFO hdf.hdf5lib.H5 - HDF5 library: 
2852 [main] INFO hdf.hdf5lib.H5 -  successfully loaded.
....10%-- org.jblas INFO Starting temp DLL cleanup task.
..-- org.jblas INFO Deleted 4 unused temp DLL libraries from C:\Users\jepir\AppData\Local\Temp
..20%....30%....40%....50%....60%....70%....80%....90% done.

snaphu v2.0.4
2

In [17]:
# Get Stage 6 (Unwrapped files Output files)
Stage6_OutputPaths = []
for Path in os.listdir(Output_Directory_6):
    if Path.endswith('.dim'):
        Stage6_OutputPaths.append(Output_Directory_6 + Path)

In [18]:
# Count time
Time_Process_Start = time.perf_counter()
# Execute Stage 7 - Terrain Correction & Phase to Height
for Path in Stage6_OutputPaths:
    Time_Process_Start_Command = time.perf_counter()
    Sub_Swath    = Path.split('/')[-1].split('_')[-1].split('.')[0]
    Polarisation = Path.split('/')[-1].split('_')[-2]
    Input_Path_1 = Path
    Output_Path  = Output_Directory_7 + Path.split('/')[-1].rsplit('_', 2)[0] + '_tc_' + Polarisation + '_' + Sub_Swath + '.dim'
    print(Output_Path)
    print('------------------')
    #Execute Command for Stage 3
    #command = 'gpt 7_S1-SLC_TerrainCorrection_EPSG25830.xml -PInput_1=\"' + Input_Path_1 + '\" -POutput=\"' + Output_Path + '\"'
    command = 'gpt 7_S1-SLC_TerrainCorrection_EPSG3763.xml -PInput_1=\"' + Input_Path_1 + '\" -POutput=\"' + Output_Path + '\"'
    # Start process
    process = subprocess.Popen(
        command,
        shell=True,
        cwd=SNAP_Engine_Path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge stdout and stderr
        text=True
    )
    # Stream output line by line
    for line in process.stdout:
        print(line, end="")   # already has \n, so no extra newline
    print('------------------')
    # Compute processing time
    Time_Process_Ends = time.perf_counter()
    Processing_Time   = Time_Process_Ends - Time_Process_Start_Command
    # Convert to h:m:s
    hours, rem       = divmod(Processing_Time, 3600)
    minutes, seconds = divmod(rem, 60)
    print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")
# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

C:/Users/jepir/Desktop/S1-SLC/Process_PortugalNorth_A/STAGE_7/S1A_20200609_20200621_CD18_Stack_esd_ifg_deb_ml_flt_unw_tc_VH_IW1.dim
------------------
OpenJDK 64-Bit Server VM warning: Options -Xverify:none and -noverify were deprecated in JDK 13 and will likely be removed in a future release.
INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
Executing processing graph
....INFO: org.esa.snap.core.dataop.dem.ElevationFile: http retrieving http://step.esa.int/auxdata/dem/SRTMGL1/N41W010.SRTMGL1.hgt.zip
10%....20%....30%....40%INFO: org.esa.snap.core.dataop.dem.ElevationFile: http retrieving http://step.esa.int/auxdata/dem/SRTMGL1/N40W010.SRTMGL1.hgt.zip
....50%....60%....70%....80%....90% done.
------------------
Elapsed time: 00:00:40.26
C:/Users/jepir/Desktop/S1-SLC/Process_PortugalNorth_A/STAGE_7/S1A_20200609_

In [19]:
# Get Stage 7 (Terrain Corrected files)
Stage7_OutputPaths = []
for Path in os.listdir(Output_Directory_7):
    if Path.endswith('.dim'):
        Stage7_OutputPaths.append(Output_Directory_7 + Path)

In [20]:
# Get Products for Temporal Composite
Composites_Dict = dict()
# Organise products for process
for Path in Stage7_OutputPaths:
    Sub_Swath    = Path.split('/')[-1].split('_')[-1].split('.')[0]
    Polarisation = Path.split('/')[-1].split('_')[-2]
    Key          = Sub_Swath + '_' + Polarisation
    if not Composites_Dict.get(Key):
        Composites_Dict[Key] = []
    Composites_Dict[Key].append(Path)

In [21]:
# Count time
Time_Process_Start = time.perf_counter()
# Execute Stage 8 - Generate Temporal Composite
for Key in Composites_Dict.keys():
    Time_Process_Start_Command = time.perf_counter()
    # Define Parameters of the XML file
    Input_Paths   = ''
    Input_ID      = 1 
    Coh_Composite = '-PCoh_Composite=\"avg('
    Unw_Composite = '-PUnw_Composite=\"avg('
    for Path in Composites_Dict[Key]:
        Input_Paths   += '-PInput_' + str(Input_ID) + '=\"' + Path + '\" '
        # Get dates code
        Dates_Code     = Path.split('/')[-1].split('_')[1][6:8] + Months_Dict[Path.split('/')[-1].split('_')[1][4:6]] + Path.split('/')[-1].split('_')[1][0:4] + '_' \
                       + Path.split('/')[-1].split('_')[2][6:8] + Months_Dict[Path.split('/')[-1].split('_')[2][4:6]] + Path.split('/')[-1].split('_')[2][0:4]
        # Create Composite operations
        Band_ID = Input_ID - 2 if Input_ID != 1 else Input_ID - 1
        Coh_Composite += 'coh_' + Key + '_' + Dates_Code + '_' + str(Band_ID) + ','
        Unw_Composite += 'Unw_Phase_ifg_' + Dates_Code +  '_VH_' + str(Band_ID) + ','
        Input_ID      += 1
    Input_Paths   += '-PReference=\"' + Composites_Dict[Key][0].split('/')[-1].split('.')[0] + '\"'
    Coh_Composite  = Coh_Composite[:-1] + ')\"'
    Unw_Composite  = Unw_Composite[:-1] + ')\"'
    # Define Output Bands names
    Coh_Band = '-PBand_Name_1=\"Coh_'   + Key.split('_')[-1] + '\"'
    Unw_Band = '-PBand_Name_2=\"Phase_' + Key.split('_')[-1] + '\"'
    # Define Output Path
    Output_Path = Output_Directory_8 + Area_Code + Key + '.dim'
    # Generate Command
    #command = 'gpt 8_S1-SLC_TemporalComposite.xml ' + Input_Paths + ' ' + Coh_Band + ' ' + Coh_Composite + ' ' + Unw_Band + ' ' + Unw_Composite + \
    #          ' -POutput=\"' + Output_Path + '\"'
    command = 'gpt 8_S1-SLC_TemporalComposite_3Files.xml ' + Input_Paths + ' ' + Coh_Band + ' ' + Coh_Composite + ' ' + Unw_Band + ' ' + Unw_Composite + \
              ' -POutput=\"' + Output_Path + '\"' 
    # Start process
    process = subprocess.Popen(
        command,
        shell=True,
        cwd=SNAP_Engine_Path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge stdout and stderr
        text=True
    )
    # Stream output line by line
    for line in process.stdout:
        print(line, end="")   # already has \n, so no extra newline
    print('------------------')
    # Compute processing time
    Time_Process_Ends = time.perf_counter()
    Processing_Time   = Time_Process_Ends - Time_Process_Start_Command
    # Convert to h:m:s
    hours, rem       = divmod(Processing_Time, 3600)
    minutes, seconds = divmod(rem, 60)
    print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")
# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

OpenJDK 64-Bit Server VM warning: Options -Xverify:none and -noverify were deprecated in JDK 13 and will likely be removed in a future release.
INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
Executing processing graph
2685 [main] INFO hdf.hdflib.HDFLibrary - HDF4 library: 
2687 [main] INFO hdf.hdflib.HDFLibrary -  successfully loaded.
2695 [main] INFO hdf.hdf5lib.H5 - HDF5 library: 
2701 [main] INFO hdf.hdf5lib.H5 -  successfully loaded.
....10%....20%....30%....40%....50%....60%....70%....80%....90% done.
------------------
Elapsed time: 00:01:19.59
OpenJDK 64-Bit Server VM warning: Options -Xverify:none and -noverify were deprecated in JDK 13 and will likely be removed in a future release.
INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.sna

In [22]:
# Get Stage 8 (Temporal Composite files)
Stage8_OutputPaths = []
for Path in os.listdir(Output_Directory_8):
    if Path.endswith('.dim'):
        Stage8_OutputPaths.append(Output_Directory_8 + Path)

In [23]:
# Get Polarisation Products for Merge 
Polarisations_Dict = dict()
# Organise products for process
for Path in Stage8_OutputPaths:
    Key = Path.split('/')[-1].rsplit('_', 1)[0]
    if not Polarisations_Dict.get(Key):
        Polarisations_Dict[Key] = []
    Polarisations_Dict[Key].append(Path)

In [24]:
# Count time
Time_Process_Start = time.perf_counter()
# Execute Stage 9 - GLCM Features
for Key in Polarisations_Dict.keys():
    Time_Process_Start_Command = time.perf_counter()
    # Get Input Paths
    Input_Path_1 = Polarisations_Dict[Key][0]
    Input_Path_2 = Polarisations_Dict[Key][1]
    # Define Output Path
    Output_Path  = Output_Directory_9 + Key + '.tif'
    print(Output_Path)
    print('------------------')
    #Execute Command for Stage 3
    command = 'gpt 9_S1-SLC_GLCM.xml -PInput_1=\"' + Input_Path_1 + '\" -PInput_2=\"' + Input_Path_2 + '\" -POutput=\"' + Output_Path + '\"'
    # Start process
    process = subprocess.Popen(
        command,
        shell=True,
        cwd=SNAP_Engine_Path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge stdout and stderr
        text=True
    )
    # Stream output line by line
    for line in process.stdout:
        print(line, end="")   # already has \n, so no extra newline
    print('------------------')
    # Compute processing time
    Time_Process_Ends = time.perf_counter()
    Processing_Time   = Time_Process_Ends - Time_Process_Start_Command
    # Convert to h:m:s
    hours, rem       = divmod(Processing_Time, 3600)
    minutes, seconds = divmod(rem, 60)
    print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")
# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

C:/Users/jepir/Desktop/S1-SLC/Process_PortugalNorth_A/STAGE_9/PortugalNorth_A_IW1.tif
------------------
OpenJDK 64-Bit Server VM warning: Options -Xverify:none and -noverify were deprecated in JDK 13 and will likely be removed in a future release.
INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
Executing processing graph
1663 [main] INFO hdf.hdflib.HDFLibrary - HDF4 library: 
1663 [main] INFO hdf.hdflib.HDFLibrary -  successfully loaded.
1663 [main] INFO hdf.hdf5lib.H5 - HDF5 library: 
1663 [main] INFO hdf.hdf5lib.H5 -  successfully loaded.
....10%....20%....30%....40%....50%....60%....70%....80%....90% done.
------------------
Elapsed time: 00:02:47.58
C:/Users/jepir/Desktop/S1-SLC/Process_PortugalNorth_A/STAGE_9/PortugalNorth_A_IW2.tif
------------------
OpenJDK 64-Bit Server VM warning: Options -Xverify:n

In [ ]:
Stage1_OutputPaths = []
for Path in os.listdir(Output_Directory_1 + 'IW1'):
    if Path.endswith('.dim'):
        Stage1_OutputPaths.append(Output_Directory_1 + 'IW1/' + Path)
for Path in os.listdir(Output_Directory_1 + 'IW2'):
    if Path.endswith('.dim'):
        Stage1_OutputPaths.append(Output_Directory_1 + 'IW2/' + Path)
for Path in os.listdir(Output_Directory_1 + 'IW3'):
    if Path.endswith('.dim'):
        Stage1_OutputPaths.append(Output_Directory_1 + 'IW3/' + Path)

In [ ]:
Stage1_OutputPaths

In [7]:
Stage2_OutputPaths = []
for Path in os.listdir(Output_Directory_2 + 'IW1'):
    if Path.endswith('.dim'):
        Stage2_OutputPaths.append(Output_Directory_2 + 'IW1/' + Path)
for Path in os.listdir(Output_Directory_2 + 'IW2'):
    if Path.endswith('.dim'):
        Stage2_OutputPaths.append(Output_Directory_2 + 'IW2/' + Path)
for Path in os.listdir(Output_Directory_2 + 'IW3'):
    if Path.endswith('.dim'):
        Stage2_OutputPaths.append(Output_Directory_2 + 'IW3/' + Path)

In [9]:
Stage2_OutputPaths

['C:/Users/jepir/Desktop/S1-SLC/Process_PortugalNorth_A/STAGE_2/IW2/S1A_20200808_20200820_821D_Stack_esd_IW2.dim',
 'C:/Users/jepir/Desktop/S1-SLC/Process_PortugalNorth_A/STAGE_2/IW3/S1A_20200808_20200820_821D_Stack_esd_IW3.dim']

In [ ]:
Stage3_OutputPaths = []
for Path in os.listdir(Output_Directory_3 + 'IW1'):
    if Path.endswith('.dim'):
        Stage3_OutputPaths.append(Output_Directory_3 + 'IW1/' + Path)
for Path in os.listdir(Output_Directory_3 + 'IW2'):
    if Path.endswith('.dim'):
        Stage3_OutputPaths.append(Output_Directory_3 + 'IW2/' + Path)
for Path in os.listdir(Output_Directory_3 + 'IW3'):
    if Path.endswith('.dim'):
        Stage3_OutputPaths.append(Output_Directory_3 + 'IW3/' + Path)

In [ ]:
Stage3_OutputPaths

In [ ]:
Stage4_OutputPaths = []
for Path in os.listdir(Output_Directory_4 + 'IW1'):
    if Path.endswith('.dim'):
        Stage4_OutputPaths.append(Output_Directory_4 + 'IW1/' + Path)
for Path in os.listdir(Output_Directory_4 + 'IW2'):
    if Path.endswith('.dim'):
        Stage4_OutputPaths.append(Output_Directory_4 + 'IW2/' + Path)
for Path in os.listdir(Output_Directory_4 + 'IW3'):
    if Path.endswith('.dim'):
        Stage4_OutputPaths.append(Output_Directory_4 + 'IW3/' + Path)

In [ ]:
Stage4_OutputPaths

In [ ]:
Stage5_OutputPaths = []
for Path in os.listdir(Output_Directory_5 + 'IW1'):
    if Path.endswith('.dim'):
        Stage5_OutputPaths.append(Output_Directory_5 + 'IW1/' + Path)
for Path in os.listdir(Output_Directory_5 + 'IW2'):
    if Path.endswith('.dim'):
        Stage5_OutputPaths.append(Output_Directory_5 + 'IW2/' + Path)
for Path in os.listdir(Output_Directory_5 + 'IW3'):
    if Path.endswith('.dim'):
        Stage5_OutputPaths.append(Output_Directory_5 + 'IW3/' + Path)

In [ ]:
Stage5_OutputPaths

In [ ]:
Stage5_OutputPaths_VV = []
for Path in os.listdir(Output_Directory_5 + 'IW1'):
    if Path.endswith('.dim') and 'VV' in Path:
        Stage5_OutputPaths_VV.append(Output_Directory_5 + 'IW1/' + Path)
for Path in os.listdir(Output_Directory_5 + 'IW2'):
    if Path.endswith('.dim') and 'VV' in Path:
        Stage5_OutputPaths_VV.append(Output_Directory_5 + 'IW2/' + Path)
for Path in os.listdir(Output_Directory_5 + 'IW3'):
    if Path.endswith('.dim') and 'VV' in Path:
        Stage5_OutputPaths_VV.append(Output_Directory_5 + 'IW3/' + Path)

In [ ]:
Stage5_OutputPaths_VV

In [ ]:
Stage5_OutputPaths_VH = []
for Path in os.listdir(Output_Directory_5 + 'IW1'):
    if Path.endswith('.dim') and 'VH' in Path:
        Stage5_OutputPaths_VH.append(Output_Directory_5 + 'IW1/' + Path)
for Path in os.listdir(Output_Directory_5 + 'IW2'):
    if Path.endswith('.dim') and 'VH' in Path:
        Stage5_OutputPaths_VH.append(Output_Directory_5 + 'IW2/' + Path)
for Path in os.listdir(Output_Directory_5 + 'IW3'):
    if Path.endswith('.dim') and 'VH' in Path:
        Stage5_OutputPaths_VH.append(Output_Directory_5 + 'IW3/' + Path)

In [ ]:
Stage5_OutputPaths_VH